# 01 - Data Preparation (Paper-Quality)

Prepares the CT-ORG dataset for full experimental pipeline.

## What it does
- Detects CT-ORG dataset under `/kaggle/input`
- Loads and preprocesses ALL cases (or selected subset)
- Simulates sparse acquisition for R=2, R=3, R=4
- Saves prepared `.npy` artifacts with metadata
- Reports dataset statistics for paper

## Configuration
- Set `FULL_EXPERIMENT = True` for paper experiments
- Set `FULL_EXPERIMENT = False` for quick debug

In [ ]:
# ========================== CONFIGURATION ==========================
FULL_EXPERIMENT = True   # True for paper, False for quick debug
SPARSE_RATIOS = [2, 3, 4]
SEED = 42
# ===================================================================

In [ ]:
import os
import sys
import json
import subprocess
import time
from pathlib import Path

KAGGLE_INPUT_DIR = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")
OUTPUT_ROOT = WORK_DIR / "outputs"
CHECKPOINT_ROOT = WORK_DIR / "checkpoints"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)

REPO_URL = "https://github.com/PhoenixEvo/ct-slice-interpolation-3dgs.git"
REPO_DIR = WORK_DIR / "ct-slice-interpolation-3dgs"

if not (REPO_DIR / "src").exists():
    print("Cloning repo...")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(str(REPO_DIR))
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

from src.utils.seed import set_seed
set_seed(SEED)

print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("CHECKPOINT_ROOT:", CHECKPOINT_ROOT)

In [ ]:
from pathlib import Path

def _count_files(root: Path, patterns: list) -> int:
    n = 0
    for pat in patterns:
        for p in root.rglob(pat):
            if p.is_file():
                n += 1
    return n

def guess_ctorg_root(kaggle_input_dir: Path, max_depth: int = 6) -> Path:
    if not kaggle_input_dir.exists():
        return Path("/kaggle/input/YOUR_DATASET/CT-ORG/OrganSegmentations")
    volume_patterns = ["volume-*.nii.gz", "volume-*.nii"]
    orgseg_names = ["OrganSegmentations", "OrganSegmentation"]
    best_root, best_score = None, -1
    for ds in kaggle_input_dir.iterdir():
        if not ds.is_dir():
            continue
        for name in orgseg_names:
            for p in ds.rglob(name):
                if p.is_dir() and len(p.relative_to(ds).parts) <= max_depth:
                    score = _count_files(p, volume_patterns)
                    if score > best_score:
                        best_score, best_root = score, p
        if best_root is None:
            score = _count_files(ds, volume_patterns)
            if score > best_score:
                best_score, best_root = score, ds
    if best_root is None or best_score <= 0:
        return Path("/kaggle/input/YOUR_DATASET/CT-ORG/OrganSegmentations")
    return best_root

DATASET_ROOT = guess_ctorg_root(KAGGLE_INPUT_DIR)
print("DATASET_ROOT:", DATASET_ROOT)
if DATASET_ROOT.exists():
    n_vol = _count_files(DATASET_ROOT, ["volume-*.nii.gz", "volume-*.nii"])
    n_lab = _count_files(DATASET_ROOT, ["labels-*.nii.gz", "labels-*.nii"])
    print(f"Volumes: {n_vol}, Labels: {n_lab}")

In [ ]:
from src.utils.config import load_config, update_config
from src.data.ct_org_loader import CTORGLoader

config = load_config("configs/default.yaml")
config = update_config(config, {
    "data": {
        "dataset_root": str(DATASET_ROOT),
        "output_root": str(OUTPUT_ROOT),
    }
})

loader = CTORGLoader(
    dataset_root=config["data"]["dataset_root"],
    hu_min=float(config["data"].get("hu_min", -1000)),
    hu_max=float(config["data"].get("hu_max", 1000)),
    normalize_range=tuple(config["data"].get("normalize_range", [0.0, 1.0])),
)

available_cases = loader.get_available_cases()
split = loader.get_split(
    available_cases=available_cases,
    test_cases=config["data"].get("test_cases", []),
    val_cases=config["data"].get("val_cases", []),
)

print(f"Available: {len(available_cases)}")
print(f"Train: {len(split['train'])}, Val: {len(split['val'])}, Test: {len(split['test'])}")
print(f"Test cases: {split['test']}")
print(f"Val cases: {split['val'][:5]}...")

In [ ]:
import numpy as np
from tqdm import tqdm
from src.data.sparse_simulator import SparseSimulator

# Determine which cases to prepare
if FULL_EXPERIMENT:
    cases_to_prepare = available_cases
else:
    cases_to_prepare = split["test"][:2] + split["train"][:2]

prepared_dir = OUTPUT_ROOT / "prepared"
prepared_dir.mkdir(parents=True, exist_ok=True)

dataset_stats = []

for case_idx in tqdm(cases_to_prepare, desc="Preparing cases"):
    try:
        volume, labels, metadata = loader.load_and_preprocess(case_idx)
    except Exception as e:
        print(f"  Skipping case {case_idx}: {e}")
        continue

    H, W, D = volume.shape
    stat = {
        "case_idx": int(case_idx),
        "shape": [H, W, D],
        "voxel_size": list(metadata.get("voxel_size", [])),
        "has_labels": labels is not None,
        "value_range": [float(volume.min()), float(volume.max())],
    }

    # Save volume and labels
    np.save(prepared_dir / f"volume_case{case_idx}.npy", volume)
    if labels is not None:
        np.save(prepared_dir / f"labels_case{case_idx}.npy", labels)

    # Save metadata
    safe_meta = {k: v for k, v in metadata.items() if k not in ("affine", "header")}
    with open(prepared_dir / f"metadata_case{case_idx}.json", "w") as f:
        json.dump(safe_meta, f, indent=2, default=str)

    # Simulate sparse acquisition for each ratio
    for R in SPARSE_RATIOS:
        sim = SparseSimulator(sparse_ratio=R)
        sparse = sim.simulate(volume)
        np.save(prepared_dir / f"observed_idx_case{case_idx}_R{R}.npy", sparse["observed_indices"])
        np.save(prepared_dir / f"target_idx_case{case_idx}_R{R}.npy", sparse["target_indices"])
        stat[f"observed_R{R}"] = len(sparse["observed_indices"])
        stat[f"targets_R{R}"] = len(sparse["target_indices"])

    dataset_stats.append(stat)

print(f"\nPrepared {len(dataset_stats)} cases")

In [ ]:
import pandas as pd

# Save and display dataset statistics (useful for paper)
stats_path = prepared_dir / "dataset_statistics.json"
with open(stats_path, "w") as f:
    json.dump(dataset_stats, f, indent=2)

df_stats = pd.DataFrame(dataset_stats)
print("\n=== Dataset Statistics (for paper Table) ===")
print(f"Total cases prepared: {len(df_stats)}")
print(f"\nVolume dimensions:")
shapes = df_stats["shape"].apply(lambda x: tuple(x))
print(f"  Unique shapes: {shapes.nunique()}")
print(f"  Depth range: {df_stats['shape'].apply(lambda x: x[2]).min()} - {df_stats['shape'].apply(lambda x: x[2]).max()}")

for R in SPARSE_RATIOS:
    col = f"targets_R{R}"
    if col in df_stats.columns:
        print(f"\nR={R}: avg targets/vol = {df_stats[col].mean():.0f}, total targets = {df_stats[col].sum()}")

# Save split info
split_info = {
    "train": split["train"],
    "val": split["val"],
    "test": split["test"],
    "sparse_ratios": SPARSE_RATIOS,
    "seed": SEED,
}
with open(prepared_dir / "split_info.json", "w") as f:
    json.dump(split_info, f, indent=2)

print(f"\nSplit: train={len(split['train'])}, val={len(split['val'])}, test={len(split['test'])}")
print("\nSaved to:", prepared_dir)